In [ ]:
try:
    from databricks.sdk import WorkspaceClient
except ImportError:
    !pip install databricks-sdk
    from databricks.sdk import WorkspaceClient

In [ ]:
# Importing Libraries
import time
import json
import requests
import pandas as pd
from google.colab import userdata
from databricks.sdk import WorkspaceClient
from concurrent.futures import ThreadPoolExecutor, as_completed

# Defining functions

In [ ]:
# ----- # ----- # ----- Get Jobs for Apple ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def jobs_apple(page_number=1):
    response = ''
    try:
        url = 'https://jobs.apple.com/api/v1/search'
        payload = {
            "query": "",
            "filters": {},
            "page": page_number,
            "locale": "en-us",
            "sort": "",
            "format": {
                "longDate": "MMMM D, YYYY",
                "mediumDate": "MMM D, YYYY"
            }
        }
        response = requests.post(url, json=payload)
        response_json = json.loads(response.text)
        if 'res' in response_json:
            if 'searchResults' in response_json['res']:
                return True, response_json
        else:
            return False, {'Error': response.text}
    except Exception as e:
        # print(str(e))
        # print(f"Unable to connect to Apple: {response.text}")
        return False, {'Error': response.text}


# ----- # ----- # ----- Get Job details for Apple ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def jobs_details_apple(record=''):
    retries = 3
    while retries > 0:
      response = ''
      try:
          if record['index']%500 == 0:
            print(record['index'])
          url = f'https://jobs.apple.com/api/v1/jobDetails/{record['positionId']}?locale=en-us'
          payload = {}
          headers = {}
          response = requests.request("GET", url, headers=headers, data=payload)
          response_json = json.loads(response.text)
          retries -= 1
          if 'res' in response_json:
              return response_json['res']
          else:
              raise Exception(f"Error: {response.text}")
      except Exception as e:
          print(f"{record['positionId']}")
          print(f"Error: {response.text}")
          time.sleep(60)
    return {"Error": record['positionId']}


# ----- # ----- # ----- Fetch Job List ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def fetch_jobs():
    page_number = 1
    count = 0
    job_df_list = []

    # Perform Initial API Call
    job_bool, job_json = jobs_apple(page_number)

    # Start the loop
    while len(job_json['res']['searchResults']) != 0:
        job_bool, job_json = jobs_apple(page_number)
        job_df = pd.DataFrame(job_json['res']['searchResults'])
        count += len(job_json['res']['searchResults'])
        if count%500 == 0:
          print(f'Page Number: {page_number}, Iteration Count: {len(job_json['res']['searchResults'])}, Total Count:{count}')
        job_df_list.append(job_df)
        page_number += 1

    # Concatenate the dataset and clean
    jobs = pd.concat(job_df_list, ignore_index=True)
    jobs.to_parquet('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/apple_jobs.parquet', index=False)
    # jobs = jobs[['id', 'jobSummary', 'locations', 'positionId', 'postingDate', 'postingTitle']]
    # jobs.reset_index(inplace=True, drop=False)
    print(f'Total Jobs: {len(jobs)}')
    return jobs


# ----- # ----- # ----- Fetch Job Details ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def fetch_job_details(job_df):
    results = []

    # Parallel processing
    with ThreadPoolExecutor(max_workers=100) as executor:  # adjust workers based on API rate limits
        future_to_record = {executor.submit(jobs_details_apple, row): row for _, row in job_df.iterrows()}
        for future in as_completed(future_to_record):
            results.append(future.result())
    results_df = pd.DataFrame(results)
    results_df.to_parquet('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/apple_job_details.parquet', index=False)
    return results_df

In [ ]:
# ----- # ----- # ----- Cleanup Data ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def clean_df(job, job_details, file_name):
  result = pd.merge(job, job_details, on="Id")
  job_details.insert(loc=0, column='Company', value='Apple')
  # job_details.columns = ['Company', 'Job ID', 'Title', 'Job Summary', 'Description', 'Team',
  #                   'Location', 'Job Type', 'Preferred Qualifications', 'Minimum Qualifications', 'Posting Date']
  return job_details


# final_df = clean_df(job_details_frame)
# final_df = final_df.astype(str)
# final_df.to_csv('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/apple.csv', header=True, index=False)

In [ ]:
# ----- # ----- # ----- Upload file to Databricks ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def upload_data(data_file):
  w = WorkspaceClient(
      host = userdata.get('DATABRICKS_HOST'),
      token = userdata.get('DATABRICKS_TOKEN')
  )

  volume_path = "/Volumes/workspace/jobs_raw/job_files/" + data_file.split('/')[-1]
  with open(data_file, "rb") as f:
      w.files.upload(volume_path, f)

  print(f"Successfully uploaded {data_file} to {volume_path}")

In [ ]:
# # job_frame = fetch_jobs()
# # job_details_frame = fetch_job_details(job_frame)

# job_frame = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/JobAI/apple_jobs.csv')
# job_details_frame = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/JobAI/apple_job_details.csv')

# New Architecture

In [ ]:
job_frame = fetch_jobs()
upload_data('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/apple_jobs.parquet')
job_frame.reset_index(inplace=True, drop=False)

In [ ]:
job_details_frame = fetch_job_details(job_frame)
upload_data('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/apple_job_details.parquet')